# Notebook 11: Quantum Error Correction

**Series: Quantum Computing from Ground Up**

---

## Learning Objectives

1. Understand why quantum error correction (QEC) is essential for fault-tolerant quantum computing
2. Review classical error correction and the repetition code
3. Master the 3-qubit bit-flip code and phase-flip code
4. Explore Shor's 9-qubit code and the Steane [7,1,3] code
5. Introduction to surface codes and the stabilizer formalism
6. Implement error correction circuits in Qiskit and simulate syndrome measurements

## 1. Why Quantum Error Correction Is Needed

Quantum systems are inherently fragile. Unlike classical bits that sit comfortably in states 0 or 1, qubits exist in superpositions $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ that are extremely sensitive to environmental interactions.

### 1.1 Sources of Quantum Errors

**Decoherence** arises from unwanted entanglement between the qubit and its environment:

$$|\psi\rangle_{\text{qubit}} \otimes |e_0\rangle_{\text{env}} \longrightarrow \alpha|0\rangle|e_0\rangle + \beta|1\rangle|e_1\rangle$$

When the environment states $|e_0\rangle$ and $|e_1\rangle$ become distinguishable, the qubit's coherence is lost.

**Types of noise channels:**

| Noise Type | Description | Kraus Operators |
|-----------|-------------|----------------|
| Bit-flip | $|0\rangle \leftrightarrow |1\rangle$ | $K_0 = \sqrt{1-p}\,I,\; K_1 = \sqrt{p}\,X$ |
| Phase-flip | Relative phase error | $K_0 = \sqrt{1-p}\,I,\; K_1 = \sqrt{p}\,Z$ |
| Depolarizing | Random Pauli error | $K_0 = \sqrt{1-p}\,I,\; K_i = \sqrt{p/3}\,\sigma_i$ |
| Amplitude damping | Energy relaxation ($T_1$) | $K_0 = \begin{pmatrix}1&0\\0&\sqrt{1-\gamma}\end{pmatrix},\; K_1 = \begin{pmatrix}0&\sqrt{\gamma}\\0&0\end{pmatrix}$ |

### 1.2 The No-Cloning Obstacle

Classical error correction relies on copying bits. But the **no-cloning theorem** states:

> There exists no unitary $U$ such that $U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$ for all $|\psi\rangle$.

**Proof sketch:** Suppose $U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$ and $U|\phi\rangle|0\rangle = |\phi\rangle|\phi\rangle$. Taking inner products:

$$\langle\psi|\phi\rangle = (\langle\psi|\phi\rangle)^2$$

This means $\langle\psi|\phi\rangle \in \{0, 1\}$, so cloning only works for orthogonal or identical states. $\square$

Despite this, quantum error correction is possible by encoding information into **entangled states** across multiple qubits.

## 2. Classical Error Correction Review

### 2.1 Repetition Code

The simplest classical error correcting code encodes one logical bit into $n$ physical bits by repetition:

$$0 \rightarrow 000, \quad 1 \rightarrow 111$$

This is a $[3,1,3]$ code: 3 physical bits, 1 logical bit, distance 3 (can correct 1 error).

**Decoding** uses majority voting. If we receive $010$, the majority is $0$, so we decode to $0$.

### 2.2 Linear Codes and Parity Checks

A linear code is defined by its **parity check matrix** $H$. For the 3-bit repetition code:

$$H = \begin{pmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \end{pmatrix}$$

The **syndrome** $s = Hx^T \pmod{2}$ identifies the error pattern:

| Error | Syndrome $s$ |
|-------|-------------|
| None  | $(0,0)$ |
| Bit 1 | $(1,0)$ |
| Bit 2 | $(1,1)$ |
| Bit 3 | $(0,1)$ |

The syndrome tells us *where* the error occurred without revealing the encoded information.

In [ ]:
# Classical repetition code demonstration
import numpy as np

def classical_encode(bit):
    """Encode a single bit using 3-bit repetition code."""
    return [bit, bit, bit]

def classical_syndrome(received):
    """Compute syndrome using parity check matrix."""
    H = np.array([[1, 1, 0],
                  [0, 1, 1]])
    r = np.array(received)
    return (H @ r) % 2

def classical_decode(received):
    """Decode using syndrome-based correction."""
    s = classical_syndrome(received)
    syndrome_to_error = {
        (0, 0): -1,  # No error
        (1, 0): 0,   # Error on bit 0
        (1, 1): 1,   # Error on bit 1
        (0, 1): 2    # Error on bit 2
    }
    error_pos = syndrome_to_error[tuple(s)]
    corrected = list(received)
    if error_pos >= 0:
        corrected[error_pos] ^= 1
    return corrected[0]  # Logical bit is first bit

# Test: encode 1, introduce error on bit 1
encoded = classical_encode(1)
print(f"Encoded: {encoded}")

received = [1, 0, 1]  # Error on second bit
syndrome = classical_syndrome(received)
decoded = classical_decode(received)
print(f"Received: {received}")
print(f"Syndrome: {syndrome}")
print(f"Decoded:  {decoded}")

## 3. The 3-Qubit Bit-Flip Code

### 3.1 Encoding

The quantum analogue of the repetition code protects against **bit-flip errors** ($X$ errors). The encoding maps:

$$|0\rangle_L = |000\rangle, \quad |1\rangle_L = |111\rangle$$

A general state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ becomes:

$$|\psi\rangle_L = \alpha|000\rangle + \beta|111\rangle$$

Note: this is **not** cloning! The state $\alpha|0\rangle + \beta|1\rangle$ is not copied; rather, we create an entangled three-qubit state.

### 3.2 Encoding Circuit

The encoding uses two CNOT gates with the data qubit as control:

$$|\psi\rangle|00\rangle \xrightarrow{\text{CNOT}_{12}} (\alpha|0\rangle + \beta|1\rangle)|0\rangle|0\rangle \rightarrow \alpha|000\rangle + \beta|110\rangle \xrightarrow{\text{CNOT}_{13}} \alpha|000\rangle + \beta|111\rangle$$

### 3.3 Error Detection via Syndrome Measurement

We measure **stabilizer operators** without disturbing the encoded information:

$$g_1 = Z_1 Z_2 I_3, \quad g_2 = I_1 Z_2 Z_3$$

These operators have eigenvalue $+1$ on the code space and detect errors:

| Error | $\langle g_1 \rangle$ | $\langle g_2 \rangle$ | Syndrome |
|-------|----------|----------|----------|
| $I$ | $+1$ | $+1$ | $(0,0)$ |
| $X_1$ | $-1$ | $+1$ | $(1,0)$ |
| $X_2$ | $-1$ | $-1$ | $(1,1)$ |
| $X_3$ | $+1$ | $-1$ | $(0,1)$ |

Syndrome measurement is performed using **ancilla qubits** without collapsing the encoded state.

In [ ]:
# Install dependencies
# pip install qiskit qiskit-aer matplotlib

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

def create_bit_flip_code(initial_state='0'):
    """
    Create the 3-qubit bit-flip error correction code.
    
    Qubits 0-2: data qubits
    Qubits 3-4: syndrome ancillas
    Classical bits 0-1: syndrome measurement
    Classical bit 2: logical qubit measurement
    """
    data = QuantumRegister(3, 'data')
    ancilla = QuantumRegister(2, 'ancilla')
    syndrome = ClassicalRegister(2, 'syndrome')
    result = ClassicalRegister(1, 'result')
    
    qc = QuantumCircuit(data, ancilla, syndrome, result)
    
    # Prepare initial state
    if initial_state == '1':
        qc.x(data[0])
    elif initial_state == '+':
        qc.h(data[0])
    
    qc.barrier(label='Encode')
    
    # Encoding: CNOT from qubit 0 to qubits 1, 2
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])
    
    return qc, data, ancilla, syndrome, result

# Create and display the encoding circuit
qc, data, ancilla, syndrome, result = create_bit_flip_code('+')
print("Bit-flip code encoding circuit:")
print(qc.draw(output='text'))

In [ ]:
def add_bit_flip_error(qc, data, qubit_index):
    """Simulate a bit-flip error on a specified qubit."""
    qc.barrier(label=f'Error q{qubit_index}')
    qc.x(data[qubit_index])
    return qc

def add_syndrome_measurement(qc, data, ancilla, syndrome):
    """Add syndrome measurement circuit for bit-flip code."""
    qc.barrier(label='Syndrome')
    
    # Measure Z1*Z2: use ancilla[0]
    qc.cx(data[0], ancilla[0])
    qc.cx(data[1], ancilla[0])
    
    # Measure Z2*Z3: use ancilla[1]
    qc.cx(data[1], ancilla[1])
    qc.cx(data[2], ancilla[1])
    
    # Measure ancillas
    qc.measure(ancilla[0], syndrome[0])
    qc.measure(ancilla[1], syndrome[1])
    
    return qc

def add_correction(qc, data, syndrome):
    """Apply correction based on syndrome measurement."""
    qc.barrier(label='Correct')
    
    # syndrome = 10 -> error on qubit 0
    # syndrome = 11 -> error on qubit 1  
    # syndrome = 01 -> error on qubit 2
    with qc.if_test((syndrome[0], 1)):
        with qc.if_test((syndrome[1], 0)):
            qc.x(data[0])
    with qc.if_test((syndrome[0], 1)):
        with qc.if_test((syndrome[1], 1)):
            qc.x(data[1])
    with qc.if_test((syndrome[0], 0)):
        with qc.if_test((syndrome[1], 1)):
            qc.x(data[2])
    
    return qc

# Build full error correction circuit
qc, data, ancilla, syndrome, result = create_bit_flip_code('1')
qc = add_bit_flip_error(qc, data, qubit_index=1)  # Error on qubit 1
qc = add_syndrome_measurement(qc, data, ancilla, syndrome)
qc = add_correction(qc, data, syndrome)

# Decode: reverse the encoding
qc.barrier(label='Decode')
qc.cx(data[0], data[2])
qc.cx(data[0], data[1])
qc.measure(data[0], result[0])

print("Full bit-flip error correction circuit:")
print(qc.draw(output='text', fold=120))

In [ ]:
# Simulate the bit-flip code
simulator = AerSimulator()

# Test all single-qubit errors
print("=" * 50)
print("Bit-Flip Code: Syndrome Measurement Results")
print("=" * 50)

for error_qubit in [None, 0, 1, 2]:
    qc, data, ancilla, syndrome, result = create_bit_flip_code('1')
    
    if error_qubit is not None:
        qc = add_bit_flip_error(qc, data, error_qubit)
    
    qc = add_syndrome_measurement(qc, data, ancilla, syndrome)
    qc = add_correction(qc, data, syndrome)
    
    # Decode
    qc.barrier()
    qc.cx(data[0], data[2])
    qc.cx(data[0], data[1])
    qc.measure(data[0], result[0])
    
    # Run simulation
    job = simulator.run(qc, shots=1024)
    counts = job.result().get_counts()
    
    error_str = f"X on qubit {error_qubit}" if error_qubit is not None else "No error"
    print(f"\n{error_str}:")
    for outcome, count in sorted(counts.items()):
        print(f"  Result={outcome.split()[0]}, Syndrome={outcome.split()[1]}: {count}/1024")

## 4. The 3-Qubit Phase-Flip Code

### 4.1 Phase Errors

A phase-flip error applies the $Z$ operator:

$$Z(\alpha|0\rangle + \beta|1\rangle) = \alpha|0\rangle - \beta|1\rangle$$

The key insight is that a phase flip in the $Z$-basis is a bit flip in the $X$-basis:

$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \quad |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

$$Z|+\rangle = |-\rangle, \quad Z|-\rangle = |+\rangle$$

### 4.2 Encoding

The phase-flip code encodes in the Hadamard basis:

$$|0\rangle_L = |{+}{+}{+}\rangle, \quad |1\rangle_L = |{-}{-}{-}\rangle$$

The encoding circuit: first apply the bit-flip encoding, then apply $H$ to all three qubits.

### 4.3 Stabilizers

The stabilizers for the phase-flip code are:

$$g_1 = X_1 X_2 I_3, \quad g_2 = I_1 X_2 X_3$$

These detect phase ($Z$) errors the same way $ZZ$ stabilizers detect bit-flip ($X$) errors.

In [ ]:
def create_phase_flip_code(initial_state='0'):
    """Create the 3-qubit phase-flip error correction code."""
    data = QuantumRegister(3, 'data')
    ancilla = QuantumRegister(2, 'ancilla')
    syndrome = ClassicalRegister(2, 'syndrome')
    result = ClassicalRegister(1, 'result')
    
    qc = QuantumCircuit(data, ancilla, syndrome, result)
    
    if initial_state == '1':
        qc.x(data[0])
    elif initial_state == '+':
        qc.h(data[0])
    
    qc.barrier(label='Encode')
    
    # Bit-flip encoding
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])
    
    # Switch to Hadamard basis
    qc.h(data[0])
    qc.h(data[1])
    qc.h(data[2])
    
    return qc, data, ancilla, syndrome, result

def add_phase_syndrome(qc, data, ancilla, syndrome):
    """Add syndrome measurement for phase-flip code (X-basis parity checks)."""
    qc.barrier(label='Syndrome')
    
    # Measure X1*X2: Hadamard, CNOT, Hadamard
    qc.h(data[0])
    qc.h(data[1])
    qc.cx(data[0], ancilla[0])
    qc.cx(data[1], ancilla[0])
    qc.h(data[0])
    qc.h(data[1])
    
    # Measure X2*X3
    qc.h(data[1])
    qc.h(data[2])
    qc.cx(data[1], ancilla[1])
    qc.cx(data[2], ancilla[1])
    qc.h(data[1])
    qc.h(data[2])
    
    qc.measure(ancilla[0], syndrome[0])
    qc.measure(ancilla[1], syndrome[1])
    return qc

# Test phase-flip code
print("Phase-Flip Code: Syndrome Results")
print("=" * 50)

for error_qubit in [None, 0, 1, 2]:
    qc, data, ancilla, syndrome, result = create_phase_flip_code('0')
    
    if error_qubit is not None:
        qc.barrier(label=f'Z error q{error_qubit}')
        qc.z(data[error_qubit])  # Phase-flip error
    
    qc = add_phase_syndrome(qc, data, ancilla, syndrome)
    
    job = simulator.run(qc, shots=1024)
    counts = job.result().get_counts()
    
    error_str = f"Z on qubit {error_qubit}" if error_qubit is not None else "No error"
    print(f"\n{error_str}: {counts}")

## 5. Shor's 9-Qubit Code

### 5.1 Handling Both Bit and Phase Flips

The bit-flip code corrects $X$ errors but not $Z$ errors. The phase-flip code corrects $Z$ errors but not $X$ errors. **Shor's code** combines both to correct arbitrary single-qubit errors.

An arbitrary error on a single qubit can be decomposed into Pauli operators:

$$E = c_I I + c_X X + c_Y Y + c_Z Z$$

Since $Y = iXZ$, if we can correct both $X$ and $Z$ errors, we can correct any error.

### 5.2 Encoding

Shor's code uses 9 qubits with a two-level concatenation:

**Step 1 (Phase-flip protection):** Encode in the $|\pm\rangle$ basis:
$$|0\rangle \rightarrow |{+}{+}{+}\rangle, \quad |1\rangle \rightarrow |{-}{-}{-}\rangle$$

**Step 2 (Bit-flip protection):** Encode each $|\pm\rangle$ using the repetition code:
$$|+\rangle \rightarrow \frac{|000\rangle + |111\rangle}{\sqrt{2}}, \quad |-\rangle \rightarrow \frac{|000\rangle - |111\rangle}{\sqrt{2}}$$

The logical codewords are:

$$|0\rangle_L = \frac{1}{2\sqrt{2}}(|000\rangle + |111\rangle)^{\otimes 3}$$

$$|1\rangle_L = \frac{1}{2\sqrt{2}}(|000\rangle - |111\rangle)^{\otimes 3}$$

### 5.3 Stabilizer Generators

Shor's code has 8 stabilizer generators:

**Bit-flip detection** (within each block of 3):
$$g_1 = Z_1Z_2, \quad g_2 = Z_2Z_3, \quad g_3 = Z_4Z_5, \quad g_4 = Z_5Z_6, \quad g_5 = Z_7Z_8, \quad g_6 = Z_8Z_9$$

**Phase-flip detection** (between blocks):
$$g_7 = X_1X_2X_3X_4X_5X_6, \quad g_8 = X_4X_5X_6X_7X_8X_9$$

In [ ]:
def create_shor_code():
    """
    Create Shor's 9-qubit error correction code.
    Protects against arbitrary single-qubit errors.
    """
    qr = QuantumRegister(9, 'q')
    qc = QuantumCircuit(qr)
    
    # Step 1: Phase-flip encoding
    # |0> -> |+++>, |1> -> |--->
    qc.cx(qr[0], qr[3])
    qc.cx(qr[0], qr[6])
    qc.h(qr[0])
    qc.h(qr[3])
    qc.h(qr[6])
    
    qc.barrier()
    
    # Step 2: Bit-flip encoding for each block
    # Block 1: qubits 0,1,2
    qc.cx(qr[0], qr[1])
    qc.cx(qr[0], qr[2])
    
    # Block 2: qubits 3,4,5
    qc.cx(qr[3], qr[4])
    qc.cx(qr[3], qr[5])
    
    # Block 3: qubits 6,7,8
    qc.cx(qr[6], qr[7])
    qc.cx(qr[6], qr[8])
    
    return qc, qr

qc_shor, qr_shor = create_shor_code()
print("Shor's 9-qubit encoding circuit:")
print(qc_shor.draw(output='text'))
print(f"\nCircuit depth: {qc_shor.depth()}")
print(f"Gate count: {qc_shor.count_ops()}")

## 6. The Steane [7,1,3] Code

### 6.1 CSS Codes

The **Calderbank-Shor-Steane (CSS)** construction builds quantum codes from two classical codes $C_1$ and $C_2$ with $C_2 \subseteq C_1$. The resulting quantum code has parameters:

$$[[n, k_1 - k_2, \min(d_1, d_2^\perp)]]$$

### 6.2 Steane Code

The Steane code is a $[[7,1,3]]$ CSS code based on the classical $[7,4,3]$ Hamming code. It encodes 1 logical qubit into 7 physical qubits and can correct any single-qubit error.

**Logical codewords:**

$$|0\rangle_L = \frac{1}{\sqrt{8}}(|0000000\rangle + |1010101\rangle + |0110011\rangle + |1100110\rangle$$
$$\quad\quad\quad + |0001111\rangle + |1011010\rangle + |0111100\rangle + |1101001\rangle)$$

$$|1\rangle_L = X^{\otimes 7}|0\rangle_L$$

### 6.3 Stabilizer Generators

The Steane code has 6 stabilizer generators:

**$X$-type (detect $Z$ errors):**
$$g_1 = IIIXXXX, \quad g_2 = IXXIIXX, \quad g_3 = XIXIXIX$$

**$Z$-type (detect $X$ errors):**
$$g_4 = IIIZZZZ, \quad g_5 = IZZIIZZ, \quad g_6 = ZIZIZIZ$$

### 6.4 Advantages of the Steane Code

- Uses fewer qubits than Shor's code (7 vs 9)
- Transversal implementation of all Clifford gates
- Beautiful connection to classical coding theory

In [ ]:
def create_steane_code_encoder():
    """
    Create the Steane [[7,1,3]] code encoding circuit.
    Based on the [7,4,3] Hamming code.
    """
    qr = QuantumRegister(7, 'q')
    qc = QuantumCircuit(qr)
    
    # The encoding circuit for the Steane code
    # Qubit 0 is the logical qubit
    # Step 1: Create superposition structure
    qc.h(qr[4])
    qc.h(qr[5])
    qc.h(qr[6])
    
    # Step 2: CNOT network to create code structure
    qc.cx(qr[0], qr[1])
    qc.cx(qr[0], qr[2])
    
    qc.cx(qr[6], qr[0])
    qc.cx(qr[6], qr[1])
    qc.cx(qr[6], qr[3])
    
    qc.cx(qr[5], qr[0])
    qc.cx(qr[5], qr[2])
    qc.cx(qr[5], qr[3])
    
    qc.cx(qr[4], qr[1])
    qc.cx(qr[4], qr[2])
    qc.cx(qr[4], qr[3])
    
    return qc, qr

qc_steane, qr_steane = create_steane_code_encoder()
print("Steane [7,1,3] encoding circuit:")
print(qc_steane.draw(output='text'))
print(f"\nCircuit depth: {qc_steane.depth()}")
print(f"Gate count: {qc_steane.count_ops()}")

## 7. Stabilizer Formalism

### 7.1 The Pauli Group

The **$n$-qubit Pauli group** $\mathcal{G}_n$ consists of all $n$-fold tensor products of Pauli matrices with phases $\{\pm 1, \pm i\}$:

$$\mathcal{G}_n = \{\pm 1, \pm i\} \times \{I, X, Y, Z\}^{\otimes n}$$

### 7.2 Stabilizer Group

A **stabilizer code** encoding $k$ logical qubits into $n$ physical qubits is defined by an abelian subgroup $\mathcal{S} \subset \mathcal{G}_n$ with $|\mathcal{S}| = 2^{n-k}$ such that $-I \notin \mathcal{S}$.

The **code space** is the simultaneous $+1$ eigenspace of all stabilizers:

$$\mathcal{C} = \{|\psi\rangle : g|\psi\rangle = |\psi\rangle \;\forall\; g \in \mathcal{S}\}$$

### 7.3 Error Detection

For an error $E \in \mathcal{G}_n$ and stabilizer $g \in \mathcal{S}$, there are two cases:

1. **$E$ commutes with $g$:** $[E, g] = 0 \Rightarrow$ $g$ doesn't detect $E$
2. **$E$ anticommutes with $g$:** $\{E, g\} = 0 \Rightarrow$ $g$ detects $E$ (syndrome bit = 1)

The **syndrome** is the binary vector:

$$s(E) = (s_1, s_2, \ldots, s_{n-k}), \quad \text{where } g_i E = (-1)^{s_i} E g_i$$

### 7.4 Code Distance

The **distance** $d$ of a stabilizer code is the minimum weight of an operator in $N(\mathcal{S}) \setminus \mathcal{S}$, where $N(\mathcal{S})$ is the normalizer of $\mathcal{S}$. A $[[n,k,d]]$ code can correct $\lfloor(d-1)/2\rfloor$ errors.

### 7.5 The Knill-Laflamme Conditions

A code $\mathcal{C}$ can correct error set $\{E_a\}$ if and only if:

$$\langle i_L | E_a^\dagger E_b | j_L \rangle = C_{ab} \delta_{ij}$$

for some Hermitian matrix $C_{ab}$ and all code basis states $|i_L\rangle, |j_L\rangle$.

In [ ]:
# Stabilizer formalism demonstration
# Check commutation relations for the bit-flip code stabilizers

import numpy as np
from itertools import product

# Pauli matrices
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def tensor(*matrices):
    """Compute tensor product of multiple matrices."""
    result = matrices[0]
    for m in matrices[1:]:
        result = np.kron(result, m)
    return result

def commutator(A, B):
    """Check if A and B commute. Returns 0 for commuting, nonzero otherwise."""
    return np.allclose(A @ B, B @ A)

# Bit-flip code stabilizers
g1 = tensor(Z, Z, I2)  # Z1Z2I3
g2 = tensor(I2, Z, Z)  # I2Z2Z3

# Possible errors
errors = {
    'I': tensor(I2, I2, I2),
    'X1': tensor(X, I2, I2),
    'X2': tensor(I2, X, I2),
    'X3': tensor(I2, I2, X),
    'Z1': tensor(Z, I2, I2),
    'Z2': tensor(I2, Z, I2),
    'Z3': tensor(I2, I2, Z),
}

print("Commutation Relations for Bit-Flip Code")
print("=" * 50)
print(f"{'Error':<8} {'Commutes g1?':<15} {'Commutes g2?':<15} {'Syndrome'}")
print("-" * 50)

for name, E in errors.items():
    c1 = commutator(E, g1)
    c2 = commutator(E, g2)
    s1 = 0 if c1 else 1
    s2 = 0 if c2 else 1
    print(f"{name:<8} {str(c1):<15} {str(c2):<15} ({s1},{s2})")

print("\nNote: Z errors are undetectable (commute with all stabilizers)")
print("This is why the bit-flip code only corrects X errors.")

## 8. Surface Codes Introduction

### 8.1 Why Surface Codes?

Surface codes are the leading candidate for practical quantum error correction because:
- They require only **nearest-neighbor** interactions on a 2D lattice
- They have a relatively **high error threshold** ($\sim 1\%$)
- Syndrome extraction uses only CNOT, $H$, and measurement

### 8.2 The Toric Code

The **toric code** is defined on an $L \times L$ lattice on a torus with qubits on edges. For each vertex $v$ and face $f$:

$$A_v = \prod_{e \ni v} X_e \quad (\text{vertex operators})$$

$$B_f = \prod_{e \in \partial f} Z_e \quad (\text{face operators})$$

These all commute because each vertex operator shares an even number of edges with each face operator.

### 8.3 Logical Operators

Logical operators are **non-contractible loops** on the torus:
- $\bar{X}$: chain of $X$ operators forming a non-contractible loop
- $\bar{Z}$: chain of $Z$ operators forming a non-contractible loop (dual lattice)

The code distance $d = L$ (the linear size of the lattice).

### 8.4 Error Threshold

If the physical error rate $p$ is below the **threshold** $p_\text{th}$, the logical error rate decreases exponentially with code distance:

$$p_L \sim \left(\frac{p}{p_{\text{th}}}\right)^{d/2}$$

For the surface code, $p_{\text{th}} \approx 1\%$ under circuit-level noise.

In [ ]:
# Visualize the surface code lattice structure
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

L = 3  # Lattice size

# Draw edges (qubits)
for i in range(L):
    for j in range(L):
        # Horizontal edges
        ax.plot([j, j+1], [i, i], 'b-', linewidth=2)
        ax.plot([(j+j+1)/2], [i], 'bs', markersize=10)  # Qubit marker
        
        # Vertical edges
        ax.plot([j, j], [i, i+1], 'b-', linewidth=2)
        ax.plot([j], [(i+i+1)/2], 'bs', markersize=10)  # Qubit marker

# Draw last row/col edges
for j in range(L):
    ax.plot([j, j+1], [L, L], 'b-', linewidth=2)
    ax.plot([(j+j+1)/2], [L], 'bs', markersize=10)
for i in range(L):
    ax.plot([L, L], [i, i+1], 'b-', linewidth=2)
    ax.plot([L], [(i+i+1)/2], 'bs', markersize=10)

# Draw vertex operators (A_v) - red stars
for i in range(L+1):
    for j in range(L+1):
        ax.plot(j, i, 'r*', markersize=15, label='Vertex (A_v)' if i==0 and j==0 else '')

# Draw face operators (B_f) - green circles
for i in range(L):
    for j in range(L):
        ax.plot(j+0.5, i+0.5, 'go', markersize=15, label='Face (B_f)' if i==0 and j==0 else '')

# Highlight one vertex operator
v_x, v_y = 1, 1
for dx, dy in [(0.5, 0), (-0.5, 0), (0, 0.5), (0, -0.5)]:
    ax.annotate('', xy=(v_x+dx, v_y+dy), xytext=(v_x, v_y),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.set_xlim(-0.5, L+0.5)
ax.set_ylim(-0.5, L+0.5)
ax.set_aspect('equal')
ax.set_title('Surface Code Lattice (d=3)', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlabel('Blue squares = qubits, Red stars = vertex ops, Green circles = face ops')
plt.tight_layout()
plt.show()

In [ ]:
# Simulate error rates: logical vs physical for surface-code-like scaling
import numpy as np
import matplotlib.pyplot as plt

p_physical = np.linspace(0.001, 0.02, 100)
p_threshold = 0.01

fig, ax = plt.subplots(figsize=(10, 6))

for d in [3, 5, 7, 9, 11]:
    # Approximate logical error rate
    p_logical = 0.1 * (p_physical / p_threshold) ** (d / 2)
    p_logical = np.minimum(p_logical, 0.5)  # Cap at 0.5
    ax.semilogy(p_physical * 100, p_logical, label=f'd = {d}', linewidth=2)

ax.axvline(x=p_threshold * 100, color='k', linestyle='--', alpha=0.5, label='Threshold')
ax.set_xlabel('Physical Error Rate (%)', fontsize=12)
ax.set_ylabel('Logical Error Rate (per round)', fontsize=12)
ax.set_title('Surface Code: Logical Error Rate vs Physical Error Rate', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-15, 1)
plt.tight_layout()
plt.show()

print("Below threshold: increasing d exponentially suppresses logical errors.")
print("Above threshold: increasing d makes things WORSE.")

## 9. Complete Error Correction Simulation

Let us now run a full simulation of the 3-qubit bit-flip code with realistic noise to demonstrate that error correction improves fidelity.

In [ ]:
from qiskit_aer.noise import NoiseModel, pauli_error

def simulate_with_noise(use_correction=True, p_error=0.1, shots=10000):
    """
    Simulate encoding |1> with a noisy channel.
    Returns fraction of correct outcomes.
    """
    if use_correction:
        # With error correction
        data = QuantumRegister(3, 'data')
        ancilla = QuantumRegister(2, 'anc')
        syn = ClassicalRegister(2, 'syn')
        res = ClassicalRegister(1, 'res')
        qc = QuantumCircuit(data, ancilla, syn, res)
        
        # Prepare |1>
        qc.x(data[0])
        
        # Encode
        qc.cx(data[0], data[1])
        qc.cx(data[0], data[2])
        
        # Noisy channel (applied via noise model)
        qc.id(data[0])
        qc.id(data[1])
        qc.id(data[2])
        
        # Syndrome measurement
        qc.cx(data[0], ancilla[0])
        qc.cx(data[1], ancilla[0])
        qc.cx(data[1], ancilla[1])
        qc.cx(data[2], ancilla[1])
        qc.measure(ancilla[0], syn[0])
        qc.measure(ancilla[1], syn[1])
        
        # Correction
        with qc.if_test((syn[0], 1)):
            with qc.if_test((syn[1], 0)):
                qc.x(data[0])
        with qc.if_test((syn[0], 1)):
            with qc.if_test((syn[1], 1)):
                qc.x(data[1])
        with qc.if_test((syn[0], 0)):
            with qc.if_test((syn[1], 1)):
                qc.x(data[2])
        
        # Decode and measure
        qc.cx(data[0], data[2])
        qc.cx(data[0], data[1])
        qc.measure(data[0], res[0])
    else:
        # Without error correction - single qubit
        qr = QuantumRegister(1, 'q')
        res = ClassicalRegister(1, 'res')
        qc = QuantumCircuit(qr, res)
        qc.x(qr[0])
        qc.id(qr[0])  # Noisy identity
        qc.measure(qr[0], res[0])
    
    # Create noise model
    noise_model = NoiseModel()
    error = pauli_error([('X', p_error), ('I', 1 - p_error)])
    noise_model.add_all_qubit_quantum_error(error, ['id'])
    
    simulator = AerSimulator(noise_model=noise_model)
    job = simulator.run(qc, shots=shots)
    counts = job.result().get_counts()
    
    # Count correct outcomes (should be |1>)
    correct = 0
    for outcome, count in counts.items():
        result_bit = outcome.strip().split()[-1] if use_correction else outcome.strip()
        if result_bit == '1':
            correct += count
    
    return correct / shots

# Compare with and without error correction
error_rates = np.linspace(0.01, 0.3, 15)
fidelity_uncorrected = []
fidelity_corrected = []

print("Simulating... (this may take a moment)")
for p in error_rates:
    f_unc = simulate_with_noise(use_correction=False, p_error=p, shots=5000)
    f_cor = simulate_with_noise(use_correction=True, p_error=p, shots=5000)
    fidelity_uncorrected.append(f_unc)
    fidelity_corrected.append(f_cor)
    print(f"p={p:.3f}: Uncorrected={f_unc:.4f}, Corrected={f_cor:.4f}")

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(error_rates * 100, fidelity_uncorrected, 'ro-', label='Uncorrected (1 qubit)', linewidth=2, markersize=8)
ax.plot(error_rates * 100, fidelity_corrected, 'bs-', label='3-qubit bit-flip code', linewidth=2, markersize=8)

# Theoretical curves
p_theory = np.linspace(0.01, 0.3, 100)
ax.plot(p_theory * 100, 1 - p_theory, 'r--', alpha=0.5, label='Theory: 1-p')
ax.plot(p_theory * 100, 1 - 3*p_theory**2 + 2*p_theory**3, 'b--', alpha=0.5, 
        label='Theory: 1-3p²+2p³')

ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('Physical Bit-Flip Error Rate (%)', fontsize=12)
ax.set_ylabel('Probability of Correct Outcome', fontsize=12)
ax.set_title('Quantum Error Correction: 3-Qubit Bit-Flip Code', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Error correction wins when p < 0.5 (the break-even point).")
print("The corrected code fails only when 2+ qubits are flipped: P_fail ~ 3p²")

## 10. Summary and Key Takeaways

### What We Learned

| Concept | Key Formula / Idea |
|---------|--------------------|
| No-cloning theorem | Cannot copy unknown quantum states |
| Bit-flip code $[[3,1,1_X]]$ | $|0\rangle_L = |000\rangle$, stabilizers $Z_1Z_2, Z_2Z_3$ |
| Phase-flip code $[[3,1,1_Z]]$ | $|0\rangle_L = |{+}{+}{+}\rangle$, stabilizers $X_1X_2, X_2X_3$ |
| Shor's code $[[9,1,3]]$ | Concatenation of bit-flip + phase-flip |
| Steane code $[[7,1,3]]$ | CSS code from Hamming code, transversal Cliffords |
| Stabilizer formalism | Code space = $+1$ eigenspace of stabilizer group |
| Surface codes | 2D lattice, $\sim 1\%$ threshold, nearest-neighbor |
| Knill-Laflamme | $\langle i_L|E_a^\dagger E_b|j_L\rangle = C_{ab}\delta_{ij}$ |

### Looking Ahead

In the next notebook, we will explore **Quantum Walks** — the quantum analogue of random walks that provide quadratic speedups for search and graph problems.

---

*Notebook 11 of the Quantum Computing from Ground Up series.*